In [0]:
dbutils.widgets.text("env", "dev", "env")
dbutils.widgets.text("param_name", "ingestion", "param name")
dbutils.widgets.text("process_name", "gitPluse_ingestion", "process name")
dbutils.widgets.text("process_date", "16-07-2026", "process date")


In [0]:
global env, param_name, procss_name, process_date,workspace, metadata_schema
env = dbutils.widgets.get("env")
param_name = dbutils.widgets.get("param_name")
procss_name = dbutils.widgets.get("process_name")
process_date = dbutils.widgets.get("process_date")
workspace = 'workspace'
metadata_schema = f"metadata_{env}"
print(env)
print(param_name)
print(procss_name)
print(process_date)
print(workspace)
print(metadata_schema)

In [0]:

metadata_tbl = ["audit_log","dp_batch","dp_calender","dp_metadata","param_table","table_column_metadata"]

def run_metadata_notebook(env, workspace, timeout=300):
    result = dbutils.notebook.run(
        "../metadata/metadata_table",
        timeout,
        {
            "env": f"{env}",
            "workspace": f"{workspace}"
        }
    )
    return result

In [0]:
# from pyspark.sql.utils import AnalysisException

missing_items = []

# Check if schema exists
schemas = [row['databaseName'] for row in spark.sql(f"SHOW DATABASES in {workspace}").collect()]
print(schemas)
if metadata_schema not in schemas:
    missing_items.append(f"Schema '{metadata_schema}'")

# Check if tables exist
else:
    tables = [row['tableName'] for row in spark.sql(f"SHOW TABLES IN {metadata_schema}").collect()]
    print(tables)
    for tbl in metadata_tbl:
        tbl = tbl+f"_{env}"
        if tbl not in tables:
            missing_items.append(f"Table '{tbl}' in schema '{metadata_schema}'")

if missing_items:
    for item in missing_items:
        print(f"Missing: {item}")
    run_metadata_notebook(env, workspace)

In [0]:
def get_param_details():
    df_param_table = spark.sql(f"select * from {metadata_schema}.param_table_{env} where param_name='{param_name}'")
    records = df_param_table.toPandas().to_dict('records')
    if not records:
        print("No records found for the given param_name.")
        return None
    return records[0]

#fetching process details from metadata table
# param_details = get_param_details()
# print(param_details)

In [0]:
def get_process_details():
    df_process_details = spark.sql(
        f"select * from {metadata_schema}.dp_metadata_{env} where param_name='{param_name}' and process_name = '{procss_name}'"
    )
    records = df_process_details.toPandas().to_dict('records')
    if not records:
        print("No records found for the given param_name and process_name.")
        return None
    return records

# process_details = get_process_details()
# print(process_details)

In [0]:
def get_daily_run_status():
    df_daily_run_status = spark.sql(
        f"select * from {metadata_schema}.dp_batch_{env} where param_name='{param_name}' and process_name = '{procss_name}' and process_date = cast('{process_date}' as int)"
    )
    records = df_daily_run_status.toPandas().to_dict('records')
    if not records:
        print("No records found for the given param_name and process_name.")
        return None
    return records

# daily_run_status = get_daily_run_status()
# print(daily_run_status)

In [0]:
try:
    param_details = get_param_details()
    spark.sql(
        f"""
        INSERT INTO {metadata_schema}.audit_log_{env} 
        VALUES ('{param_name}', '{procss_name}', 'param_table_check','param_table_check','param_table_check','param_table_check',cast('{process_date}' as int), current_timestamp(),'SUCCESS','No logs generated')
        """
    )
except Exception as e:
    error_msg = str(e).replace("'", "''")
    spark.sql(
        f"""
        INSERT INTO {metadata_schema}.audit_log_{env} 
        VALUES ('{param_name}', '{procss_name}', 'param_table_check','param_table_check','param_table_check','param_table_check',cast('{process_date}' as int), current_timestamp(),'FAILED','error: {error_msg}')
        """
    )
    print(f"Error: {e}")
    

In [0]:
try:
   process_details = get_process_details()
   spark.sql(
        f"""
        INSERT INTO {metadata_schema}.audit_log_{env} 
        VALUES ('{param_name}', '{procss_name}', 'dp_metdata_table_check','dp_metdata_table_check','dp_metdata_table_check','dp_metdata_table_check',cast('{process_date}' as int), current_timestamp(),'SUCCESS','No logs generated')
        """
    )
except Exception as e:
    error_msg = str(e).replace("'", "''")
    spark.sql(
        f"""
        INSERT INTO {metadata_schema}.audit_log_{env} (param_name, process_name, process_date, process_status)
        VALUES ('{param_name}', '{procss_name}', 'dp_metdata_table_check','dp_metdata_table_check','dp_metdata_table_check','dp_metdata_table_check',cast('{process_date}' as int), current_timestamp(),'FAILED','error: {error_msg}')
        """
    )
    print(f"Error: {e}")

In [0]:
try:
   daily_run_status = get_daily_run_status()
   spark.sql(
        f"""
        INSERT INTO {metadata_schema}.audit_log_{env} 
        VALUES ('{param_name}', '{procss_name}', 'dp_batch_table_check','dp_batch_table_check','dp_batch_table_check','dp_batch_table_check',cast('{process_date}' as int), current_timestamp(),'SUCCESS','No logs generated')
        """
    )
except Exception as e:
    error_msg = str(e).replace("'", "''")
    spark.sql(
        f"""
        INSERT INTO {metadata_schema}.audit_log_{env} 
        VALUES ('{param_name}', '{procss_name}', 'dp_batch_table_check','dp_batch_table_check','dp_batch_table_check','dp_batch_table_check',cast('{process_date}' as int), current_timestamp(),'FAILED','error: {error_msg}')
        """
    )
    print(f"Error: {e}")

In [0]:
%sql
select * from metadata_dev.audit_log_dev